# Team GORDOBOB
Austin Jia (adj2484)\
Gordon Lee (gl23578)\
Bill Ma (bm)\
David Zhang (dz)

# Setup

In [1]:
# Imports
from typing import Tuple
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
"""
Globals
"""
# Data paths
TRAIN_PATH = "./data/cattle_data_train.csv"
TEST_PATH = "./data/cattle_data_test.csv"

# Debug
DEBUG = False

# Data Cleaning
LABEL_COL = 'Milk_Yield_L'
DROP_FEATURES = ['Cattle_ID', 'Milk_Yield_L', 'Feed_Quantity_kg',
                 'Farm_ID',              # Probably too many options for one-hot
                 'Feed_Quantity_lb',     # TODO: Missing 10k, should impute
                 'Housing_Score']        # TODO: Missing 6k, should impute

# Feature Engineering
ONE_HOT_FEATURES = ['Breed', 'Climate_Zone', 'Management_System',
                    'Feed_Type', 'Lactation_Stage']

# Model
CV = 5
PCA__N_COMPONENTS = list (range (20, 26))
KNN__N_NEIGHBORS = list (range (25, 26))

# Data Exploration

In [3]:
"""
LABELS:
Cattle_ID, Breed, Climate_Zone, Management_System, Age_Months, Weight_kg,
Parity, Lactation_Stage, Days_in_Milk, Feed_Type, Feed_Quantity_kg,
Feeding_Frequency, Water_Intake_L, Walking_Distance_km, Grazing_Duration_hrs,
Rumination_Time_hrs, Resting_Hours, Ambient_Temperature_C, Humidity_percent,
Housing_Score, FMD_Vaccine, Brucellosis_Vaccine, HS_Vaccine, BQ_Vaccine,
Anthrax_Vaccine, IBR_Vaccine, BVD_Vaccine, Rabies_Vaccine,
Previous_Week_Avg_Yield, Body_Condition_Score, Milking_Interval_hrs, Date,
Farm_ID, Feed_Quantity_lb, Mastitis, Milk_Yield_L

NOTES:
- Feed_Quanitity_lb and Feed_Quantity_kg are redundant
- Some others may be worth combining (Temperature + Humidity, Rest + Rumination,
                                      Vaccines)
- Date may be condensable into season or month
- Non-numeric features: Breed, Climate_Zone, Management_System, Lactation_Stage,
                        Feed_Type, Date, Farm_ID
"""
data = pd.read_csv (TRAIN_PATH)
data.head ()

# Count NaNs
print ("Missing Counts:")
for col in data.columns:
    missing_count = data[col].isna ().sum ()
    if missing_count:
        print(f"{col}: {missing_count}")

Missing Counts:
Feed_Quantity_kg: 10481
Housing_Score: 6279
Feed_Quantity_lb: 10481


# Data Cleaning

In [4]:
def clean_data (
    raw_data: pd.DataFrame
) -> Tuple[np.ndarray, pd.DataFrame]:
    """
    Drop IDs, redundant features, and labels
    """
    labels = raw_data[LABEL_COL].values.ravel ()
    cleaned_data = raw_data.drop (DROP_FEATURES,
                                  axis = 1)
    # TODO: Impute?
    return labels, cleaned_data

labels, cleaned_data = clean_data (data)

# Feature Engineering

In [5]:
def engineer_data (
    cleaned_data: pd.DataFrame
) -> pd.DataFrame:
    """
    Feature Engineering
    """
    # Convert date to monthly circular representation
    months = pd.to_datetime (cleaned_data['Date']).dt.month
    engineered_data = cleaned_data.drop (columns = ['Date'])
    engineered_data['Month_sin'] = np.sin (2 * np.pi * months / 12)
    engineered_data['Month_cos'] = np.cos (2 * np.pi * months / 12)

    # One Hot encode
    engineered_data = pd.get_dummies (engineered_data,
                                      columns = ONE_HOT_FEATURES,
                                      drop_first = True)

    return engineered_data

feature_engineer = FunctionTransformer (engineer_data)


# Model

In [7]:
# Split
train_set, test_set, train_labels, test_labels = \
    train_test_split (cleaned_data, 
                      labels,
                      train_size = 1 - (1 / CV),
                      random_state = 0)

# Pipeline
pipeline = Pipeline ([('f_eng', feature_engineer),
                      ('scaler', StandardScaler ()), 
                      ('pca', PCA ()),
                      ('knn', KNeighborsRegressor (weights = 'distance'))])

# Grid Search (Single CV)
param_grid = {'pca__n_components': PCA__N_COMPONENTS,
              'knn__n_neighbors': KNN__N_NEIGHBORS}
gs = GridSearchCV (estimator = pipeline,
                   param_grid = param_grid,
                   cv = CV)
gs.fit (train_set, train_labels)

# Evaluate
print ("Best parameters:", gs.best_params_)
print ("Best CV score:", gs.best_score_)
preds = gs.predict (test_set)
rmse = np.sqrt (mean_squared_error (test_labels, preds))
print("Test RMSE:", rmse)

Best parameters: {'knn__n_neighbors': 25, 'pca__n_components': 25}
Best CV score: 0.0357683167275354
Test RMSE: 5.261152597758121


# Predict

In [8]:
# Build final model with all training data
final_model = gs.best_estimator_.fit (cleaned_data, labels)

# Predict
data = pd.read_csv (TRAIN_PATH)
_, data = clean_data (data)
predictions = final_model.predict (data)

# Save
out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (predictions) + 1),
                          'Milk_Yield_L': predictions})
out_data.to_csv ('predictions.csv', index = False)